# V19.1-P0 — نوت‌بوک جامع: آزمون شناسایی‌پذیری (Jacobian/SVD/Cramér–Rao) + PINN دوشاخه‌ی اصلاح‌شده + کنترل کلاسیک

**ترتیب سلول‌ها:** ۱ راه‌اندازی → ۲ پیکربندی → ۳ حل‌گر مرجع → ۴ دیتاست → **۵ آزمون P0-1 (شناسایی‌پذیری)** → ۶ معماری → ۷ هزینه/Adam → ۸ L-BFGS → ۹ ارزیابی → ۱۰ ارکستراسیون (بازوها × seedها) → ۱۱ (اختیاری) کنترل کلاسیک P1.

**قاعده‌ی کار:** ابتدا `RUN_MODE="smoke"` با همه‌ی سلول‌ها؛ سپس سلول ۵ در `medium`؛ طراحی (T, U₀) را طبق جدول تصمیم تثبیت کنید؛ سپس بازوی کنترل؛ و فقط در صورت قبولی معیار پذیرش، بازوهای فقط-h.
جزئیات کامل در `RUNBOOK_FA.md`.

In [1]:
# ============================================================
# Cell 1 — راه‌اندازی و کلیدهای اجرا
# ============================================================
import os, time, json, pickle
import numpy as np
import jax, jax.numpy as jnp
from jax import lax
import matplotlib
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)
PATCH_LEVEL = "V19.1-P0 / identifiability + scaled-PINN + IC/BC"
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)

# ---------- کلیدهای اصلی ----------
RUN_MODE = "smoke"              # smoke | dev | medium | full
RUN_IDENTIFIABILITY = True      # سلول ۵ (P0-1)
RUN_TRAINING = True             # سلول ۱۰
TRAIN_ARMS = ["control"]        # زیرمجموعه‌ی: control, honly_ww, honly_wo, honly_wd
SEEDS = [0]                     # برای H1: حداقل [0,1,2]
RUN_CLASSICAL = False           # سلول ۱۱ (P1: Tikhonov + L-BFGS بدون شبکه)

# ---------- طراحی دوقلو: پس از اجرای سلول ۵ این دو عدد را با جدول تصمیم تنظیم کنید ----------
DESIGN_T_HOURS = 5.0            # طراحی فعلی (baseline) = ۵ ساعت
DESIGN_U0 = 0.15                # دامنه‌ی سرعت مرز غربی (m/s)

_RUN_MODE_TABLE = {
    "smoke":  dict(nx=12, ny=6,  n_coll=256,  n_obs=15, adam_ep1=50,    adam_ep2=50,    lbfgs_iters=20,  id_Ts=(5, 12),         id_U0s=(0.15, 0.5)),
    "dev":    dict(nx=24, ny=12, n_coll=2000, n_obs=40, adam_ep1=5000,  adam_ep2=5000,  lbfgs_iters=300, id_Ts=(5, 12, 24),     id_U0s=(0.15, 0.5, 1.0)),
    "medium": dict(nx=24, ny=12, n_coll=2000, n_obs=40, adam_ep1=15000, adam_ep2=15000, lbfgs_iters=500, id_Ts=(5, 12, 24, 48), id_U0s=(0.15, 0.5, 1.0)),
    "full":   dict(nx=48, ny=24, n_coll=6000, n_obs=80, adam_ep1=8000,  adam_ep2=8000,  lbfgs_iters=1000, id_Ts=(5, 12, 24, 48), id_U0s=(0.15, 0.5, 1.0)),
}
RC = _RUN_MODE_TABLE[RUN_MODE]
print(f"[Cell 1] {PATCH_LEVEL} | RUN_MODE={RUN_MODE} -> {RC}")
print(f"[Cell 1] design: T={DESIGN_T_HOURS} h, U0={DESIGN_U0} m/s | arms={TRAIN_ARMS} | seeds={SEEDS}")
print(f"[Cell 1] jax devices: {jax.devices()}")

SEED = 0
_key = jax.random.PRNGKey(SEED)
def set_seed(seed):
    global _key
    _key = jax.random.PRNGKey(seed)
def next_key():
    global _key
    _key, sub = jax.random.split(_key)
    return sub

[Cell 1] V19.1-P0 / identifiability + scaled-PINN + IC/BC | RUN_MODE=smoke -> {'nx': 12, 'ny': 6, 'n_coll': 256, 'n_obs': 15, 'adam_ep1': 50, 'adam_ep2': 50, 'lbfgs_iters': 20, 'id_Ts': (5, 12), 'id_U0s': (0.15, 0.5)}
[Cell 1] design: T=5.0 h, U0=0.15 m/s | arms=['control'] | seeds=[0]
[Cell 1] jax devices: [CudaDevice(id=0), CudaDevice(id=1)]


In [2]:
# ============================================================
# Cell 2 — پیکربندی: فیزیک، شبکه‌بندی، مقیاس‌ها، annealing
# ============================================================
NX, NY = RC["nx"], RC["ny"]
LX, LY = 150e3, 50e3
DX, DY = LX / NX, LY / NY
G, H0, F0 = 9.81, 80.0, 6.5e-5
U0 = DESIGN_U0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
DT = 60.0
N_STEPS = int(round(DESIGN_T_HOURS * 3600.0 / DT))
T_TOTAL = N_STEPS * DT

_c = float(np.sqrt(G * H0))
_cfl = DT * (_c + U0) / min(DX, DY)
print(f"[Cell 2] N_STEPS={N_STEPS} (T={T_TOTAL/3600:.1f} h)  CFL={_cfl:.3f}")
assert _cfl < 0.9, "CFL ناپایدار: DT را کاهش دهید"

# مقیاس‌های بی‌بعدسازی (P0-2 ب)
A_REF = U0 * OMEGA            # شتاب جزرومدی  [m/s^2]
C_REF = H0 * U0 / LX          # مقیاس h_t و H0*div(u)  [m/s]
U_SC  = U0                    # مقیاس سرعت
H_SC  = U0 * H0 / _c          # مقیاس ارتفاع (موج پیش‌رونده)
print(f"[Cell 2] A_REF={A_REF:.3e}  C_REF={C_REF:.3e}  U_SC={U_SC}  H_SC={H_SC:.3f}")

VELOCITY_REG_CAP_MPS = 6.0
N_ZONES = 2
K_MIN, K_MAX = 3.0, 25.0
K_ANNEAL_EPOCHS = RC["adam_ep2"]
def k_schedule(epoch_in_phase2):
    frac = min(max(epoch_in_phase2 / K_ANNEAL_EPOCHS, 0.0), 1.0)
    return K_MIN + (K_MAX - K_MIN) * frac

CF_BACKGROUND = 2.5e-3
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
ZONE_SEPARATION_WEIGHT = 5.0
ZONE_SEPARATION_MARGIN = 1.5      # در واحد 1e-3

WARM_TRANSITION_EPOCHS = 200
MOMENT_DECAY_RATE = 0.97

WIND_AMPLITUDE = 3e-6
NU_DIFF = 5.0
N_COLLOCATION = RC["n_coll"]
N_OBS_PTS = RC["n_obs"]
OBS_NOISE_STD_FRAC = 0.03

ADAM_EPOCHS_PHASE1 = RC["adam_ep1"]
ADAM_EPOCHS_PHASE2 = RC["adam_ep2"]
LBFGS_MAXITER = RC["lbfgs_iters"]
PHYS_WEIGHT_PHASE1 = 0.01
PHYS_WEIGHT_PHASE2 = 1.0
ICBC_WEIGHT = 1.0
USE_ICBC = True           # شرایط اولیه/مرزی در PINN (در V19.0 غایب بود)

[Cell 2] N_STEPS=300 (T=5.0 h)  CFL=0.203
[Cell 2] A_REF=2.108e-05  C_REF=8.000e-05  U_SC=0.15  H_SC=0.428


In [3]:
# ============================================================
# Cell 3 — حل‌گر مرجع (Arakawa C) با u0 صریح + forward_obs (بدون ذخیره‌ی کل مسیر)
# ============================================================
def interp_v_to_u(v):
    v_pad = jnp.pad(v, ((1, 1), (0, 0)), mode="edge")
    v_center = 0.5 * (v_pad[:, :-1] + v_pad[:, 1:])
    return 0.5 * (v_center[:-1, :] + v_center[1:, :])

def interp_u_to_v(u):
    u_center = 0.5 * (u[:-1, :] + u[1:, :])
    u_pad = jnp.pad(u_center, ((0, 0), (1, 1)), mode="edge")
    return 0.5 * (u_pad[:, :-1] + u_pad[:, 1:])

def cf_at_u(Cf):
    return Cf[jnp.clip(jnp.arange(NX + 1), 0, NX - 1), :]
def cf_at_v(Cf):
    return Cf[:, jnp.clip(jnp.arange(NY + 1), 0, NY - 1)]

_yy_u = (jnp.arange(NY) + 0.5) * DY
WIND_PATTERN_U = jnp.broadcast_to(jnp.sin(2 * jnp.pi * _yy_u / LY)[None, :], (NX + 1, NY))
WIND_PATTERN_V = jnp.zeros((NX, NY + 1))

def laplacian_u(u):
    px = jnp.pad(u, ((1, 1), (0, 0)), mode="edge"); py = jnp.pad(u, ((0, 0), (1, 1)), mode="edge")
    return (px[2:, :] - 2*u + px[:-2, :]) / DX**2 + (py[:, 2:] - 2*u + py[:, :-2]) / DY**2
def laplacian_v(v):
    px = jnp.pad(v, ((1, 1), (0, 0)), mode="edge"); py = jnp.pad(v, ((0, 0), (1, 1)), mode="edge")
    return (px[2:, :] - 2*v + px[:-2, :]) / DX**2 + (py[:, 2:] - 2*v + py[:, :-2]) / DY**2

def shallow_water_step(state, t, Cf, wind_on, diff_on, u0):
    eta, u, v = state
    Cf_u, Cf_v = cf_at_u(Cf), cf_at_v(Cf)
    v_at_u, u_at_v = interp_v_to_u(v), interp_u_to_v(u)
    speed_u = jnp.sqrt(u**2 + v_at_u**2 + 1e-8)
    speed_v = jnp.sqrt(u_at_v**2 + v**2 + 1e-8)
    deta_dx = (eta[1:, :] - eta[:-1, :]) / DX
    deta_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fric_u = Cf_u / H0 * u * speed_u
    fric_v = Cf_v / H0 * v * speed_v
    wind_u = jnp.where(wind_on, WIND_AMPLITUDE * WIND_PATTERN_U, 0.0)
    wind_v = jnp.where(wind_on, WIND_AMPLITUDE * WIND_PATTERN_V, 0.0)
    diff_u = jnp.where(diff_on, NU_DIFF * laplacian_u(u), 0.0)
    diff_v = jnp.where(diff_on, NU_DIFF * laplacian_v(v), 0.0)
    u_int = u[1:-1, :] + DT * (-G*deta_dx + F0*v_at_u[1:-1, :] - fric_u[1:-1, :] + wind_u[1:-1, :] + diff_u[1:-1, :])
    u_new = jnp.concatenate([jnp.full((1, NY), u0 * jnp.cos(OMEGA * t)), u_int, jnp.zeros((1, NY))], axis=0)
    v_int = v[:, 1:-1] + DT * (-G*deta_dy - F0*u_at_v[:, 1:-1] - fric_v[:, 1:-1] + wind_v[:, 1:-1] + diff_v[:, 1:-1])
    v_new = jnp.concatenate([jnp.zeros((NX, 1)), v_int, jnp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def true_cf_field():
    xc = (jnp.arange(NX) + 0.5) * DX; yc = (jnp.arange(NY) + 0.5) * DY
    Xc, Yc = jnp.meshgrid(xc, yc, indexing="ij")
    boundary = 0.5 * LX + 0.3 * (Yc - 0.5 * LY)
    return jnp.where(Xc < boundary, CF_LOW_TRUE, CF_HIGH_TRUE)

def _zero_state():
    return (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))

def run_reference_solver(Cf, wind_on, diff_on, n_steps, u0):
    # traj[i] = حالت پس از (i+1) گام، یعنی در زمان (i+1)*DT
    def body(state, n):
        new = shallow_water_step(state, n * DT, Cf, wind_on, diff_on, u0)
        e, u, v = new
        return new, (e, 0.5 * (u[:-1, :] + u[1:, :]), 0.5 * (v[:, :-1] + v[:, 1:]))
    _, traj = lax.scan(body, _zero_state(), jnp.arange(n_steps))
    return traj   # (eta, u_c, v_c) هر کدام (n_steps, NX, NY)

def make_snap(n_steps):
    return np.linspace(n_steps // 4, n_steps - 1, 4).astype(int)

def forward_obs(Cf, wind_on, diff_on, n_steps, u0, snap, ix, iy, channels=("h", "u", "v")):
    # خروجی: (n_snap, n_ch, n_obs) -- فقط در زمان‌سنجه‌ها، بدون ذخیره‌ی کل مسیر (برای jacfwd/grad)
    state, done, outs = _zero_state(), 0, []
    def body(st, n):
        return shallow_water_step(st, n * DT, Cf, wind_on, diff_on, u0), None
    for s in snap:
        n_end = int(s) + 1
        state, _ = lax.scan(body, state, jnp.arange(done, n_end))
        done = n_end
        e, u, v = state
        d = {"h": e, "u": 0.5 * (u[:-1, :] + u[1:, :]), "v": 0.5 * (v[:, :-1] + v[:, 1:])}
        outs.append(jnp.stack([d[c][ix, iy] for c in channels]))
    return jnp.stack(outs)

CF_TRUE = true_cf_field()
t0 = time.time()
_eta, _u, _v = run_reference_solver(CF_TRUE, True, True, N_STEPS, U0)
_speed = np.asarray(jnp.sqrt(_u**2 + _v**2))
print(f"[Cell 3] حل‌گر مرجع {time.time()-t0:.1f}s | سرعت: میانه={np.median(_speed):.3f} max={_speed.max():.3f} m/s | std(h)={float(jnp.std(_eta)):.3f} m")
assert np.isfinite(_speed).all(), "حل‌گر واگرا شد"

[Cell 3] حل‌گر مرجع 0.9s | سرعت: میانه=0.052 max=0.190 m/s | std(h)=0.230 m


In [4]:
# ============================================================
# Cell 4 — نقاط مشاهده/کولوکیشن/IC/BC و مولد مشاهدات
# ============================================================
_rng_obs = np.random.default_rng(SEED + 1)
_flat = _rng_obs.choice(NX * NY, size=N_OBS_PTS, replace=False)
IXO, IYO = np.unravel_index(_flat, (NX, NY))
OBS_X = jnp.array((IXO + 0.5) * DX)
OBS_Y = jnp.array((IYO + 0.5) * DY)
SNAP = make_snap(N_STEPS)
OBS_T = jnp.array((SNAP + 1) * DT)     # حالت پس از s+1 گام در زمان (s+1)*DT

def make_observations(wind_on, diff_on, noise_seed):
    eta, u, v = run_reference_solver(CF_TRUE, wind_on, diff_on, N_STEPS, U0)
    clean = [np.asarray(a)[SNAP][:, IXO, IYO] for a in (eta, u, v)]
    rng = np.random.default_rng(noise_seed)
    sigs = [OBS_NOISE_STD_FRAC * float(np.std(c)) for c in clean]     # نوفه‌ی نسبی به‌ازای هر کانال (اصلاح بند ۳.۵ memo)
    obs = [jnp.array(c + rng.normal(scale=s, size=c.shape)) for c, s in zip(clean, sigs)]
    return obs, sigs, (eta, u, v)

_ks = jax.random.split(jax.random.PRNGKey(SEED + 7), 14)
COLL_X = jax.random.uniform(_ks[0], (N_COLLOCATION,), minval=0.0, maxval=LX)
COLL_Y = jax.random.uniform(_ks[1], (N_COLLOCATION,), minval=0.0, maxval=LY)
COLL_T = jax.random.uniform(_ks[2], (N_COLLOCATION,), minval=0.0, maxval=T_TOTAL)
N_ICBC = max(128, N_COLLOCATION // 8)
IC_X  = jax.random.uniform(_ks[3], (N_ICBC,), minval=DX, maxval=LX - DX)
IC_Y  = jax.random.uniform(_ks[4], (N_ICBC,), minval=0.0, maxval=LY)
BCW_Y = jax.random.uniform(_ks[5], (N_ICBC,), minval=0.0, maxval=LY)
BCW_T = jax.random.uniform(_ks[6], (N_ICBC,), minval=0.0, maxval=T_TOTAL)
BCE_Y = jax.random.uniform(_ks[7], (N_ICBC,), minval=0.0, maxval=LY)
BCE_T = jax.random.uniform(_ks[8], (N_ICBC,), minval=0.0, maxval=T_TOTAL)
BCS_X = jax.random.uniform(_ks[9], (N_ICBC,), minval=0.0, maxval=LX)
BCS_T = jax.random.uniform(_ks[10], (N_ICBC,), minval=0.0, maxval=T_TOTAL)
BCN_X = jax.random.uniform(_ks[11], (N_ICBC,), minval=0.0, maxval=LX)
BCN_T = jax.random.uniform(_ks[12], (N_ICBC,), minval=0.0, maxval=T_TOTAL)

_obs, _sigs, _ = make_observations(True, True, 999)
print(f"[Cell 4] {N_OBS_PTS} نقطه × {len(SNAP)} زمان‌سنجه | σ(h,u,v)={[f'{s:.3g}' for s in _sigs]}")
assert float(_speed.max()) < VELOCITY_REG_CAP_MPS * 1.5

[Cell 4] 15 نقطه × 4 زمان‌سنجه | σ(h,u,v)=['0.00615', '0.00235', '0.00023']


In [5]:
# ============================================================
# Cell 5 — P0-1: آزمون شناسایی‌پذیری (Jacobian/SVD + Cramér–Rao کنتراست دو-ناحیه‌ای)
# ============================================================
# پارامتر: Cf در 1e-3؛ J با 1/σ_کانال وزن‌دهی می‌شود (نوفه‌ی نسبی ۳٪ به‌ازای هر کانال).
# r_eff = #{s_i > 1} (SNR>1). Z = کنتراست حقیقی(2.5) / sd کرامر-رائو (حد پایین خطا؛ مرز معلوم فرض می‌شود => خوش‌بینانه).
def jac_scaled(T_hours, u0, cf_ref, rel_noise=OBS_NOISE_STD_FRAC, wind_on=True, diff_on=True):
    n_steps = int(round(T_hours * 3600.0 / DT))
    snap = make_snap(n_steps)
    f = lambda x: forward_obs(x.reshape(NX, NY), wind_on, diff_on, n_steps, u0, snap, IXO, IYO, ("h", "u", "v"))
    x0 = jnp.asarray(cf_ref).ravel()
    y0 = f(x0)
    sig = rel_noise * jnp.std(y0, axis=(0, 2))                    # (3,)
    J = jax.jacfwd(f)(x0) * 1e-3 / sig[None, :, None, None]        # (n_snap,3,n_obs,P)
    return J, n_steps

_ZONE_M = jnp.stack([(CF_TRUE == CF_LOW_TRUE).ravel(), (CF_TRUE == CF_HIGH_TRUE).ravel()], axis=1).astype(jnp.float64)
_SUBSETS = {"h": [0], "hu": [0, 1], "huv": [0, 1, 2]}

def identifiability_rows(Ts, U0s):
    rows = []
    for u0 in U0s:
        for T in Ts:
            for lin_name, cf_ref in (("true", CF_TRUE), ("uniform", jnp.full((NX, NY), CF_BACKGROUND))):
                t0 = time.time()
                J, n_steps = jac_scaled(T, u0, cf_ref)
                P = J.shape[-1]
                for sname, idx in _SUBSETS.items():
                    Js = J[:, idx].reshape(-1, P)
                    s = jnp.linalg.svd(Js, compute_uv=False)
                    Jz = Js @ _ZONE_M
                    F = Jz.T @ Jz
                    a = jnp.array([-1.0, 1.0])
                    sd = float(jnp.sqrt(a @ jnp.linalg.pinv(F) @ a))
                    rows.append(dict(T=T, U0=u0, lin_pt=lin_name, subset=sname, m=int(Js.shape[0]), P=int(P),
                                     r_eff=int(jnp.sum(s > 1.0)), s1=float(s[0]),
                                     cond_zone=float(jnp.linalg.cond(F)), sd_contrast=sd, Z=2.5 / sd))
                print(f"  U0={u0:<5} T={T:<3}h {lin_name:<8} ({time.time()-t0:.1f}s)  " +
                      "  ".join(f"{r['subset']}: r_eff={r['r_eff']:>3} Z={r['Z']:.2f}" for r in rows[-3:]))
    return rows

def pick_design(rows, Zmin=5.0):
    cand = [r for r in rows if r["subset"] == "huv" and r["lin_pt"] == "uniform" and r["Z"] >= Zmin]
    cand.sort(key=lambda r: (r["T"], r["U0"]))
    if not cand:
        print(f"[Cell 5] هیچ پیکربندی‌ای با Z_huv>={Zmin} در شبکه‌ی جست‌وجو نیست -> T/U0 بزرگ‌تر یا مشاهدات بیشتر لازم است.")
        return None
    best = cand[0]
    zh = [r for r in rows if r["subset"] == "h" and r["lin_pt"] == "uniform" and r["T"] == best["T"] and r["U0"] == best["U0"]][0]
    zhu = [r for r in rows if r["subset"] == "hu" and r["lin_pt"] == "uniform" and r["T"] == best["T"] and r["U0"] == best["U0"]][0]
    print(f"[Cell 5] طراحی پیشنهادی: T={best['T']} h, U0={best['U0']} | Z_huv={best['Z']:.2f}  Z_hu={zhu['Z']:.2f}  Z_h={zh['Z']:.2f}")
    return best

ID_ROWS = []
if RUN_IDENTIFIABILITY:
    print("[Cell 5] شروع آزمون P0-1 ...")
    ID_ROWS = identifiability_rows(RC["id_Ts"], RC["id_U0s"])
    with open(os.path.join(OUT_DIR, f"identifiability_{RUN_MODE}.json"), "w") as fh:
        json.dump(ID_ROWS, fh, indent=1)
    base = [r for r in ID_ROWS if r["T"] == 5 and r["U0"] == 0.15 and r["lin_pt"] == "uniform"]
    if base:
        zb = {r["subset"]: r["Z"] for r in base}
        print(f"\n[Cell 5] baseline (T=5h,U0=0.15): Z_h={zb['h']:.2f}  Z_hu={zb['hu']:.2f}  Z_huv={zb['huv']:.2f}")
        print("[Cell 5] قاعده: Z_huv<3 => فرضیه‌ی «ب» تأیید (طراحی دوقلو ضعیف است)؛ T/U0 را تغییر دهید.")
    PICK = pick_design(ID_ROWS)

[Cell 5] شروع آزمون P0-1 ...
  U0=0.15  T=5  h true     (8.7s)  h: r_eff=  0 Z=0.68  hu: r_eff=  0 Z=1.54  huv: r_eff= 10 Z=6.65
  U0=0.15  T=5  h uniform  (4.3s)  h: r_eff=  0 Z=0.68  hu: r_eff=  0 Z=1.54  huv: r_eff= 11 Z=6.73
  U0=0.15  T=12 h true     (4.2s)  h: r_eff=  0 Z=0.43  hu: r_eff=  3 Z=5.57  huv: r_eff= 17 Z=12.87
  U0=0.15  T=12 h uniform  (4.2s)  h: r_eff=  0 Z=0.43  hu: r_eff=  3 Z=5.58  huv: r_eff= 17 Z=13.39
  U0=0.5   T=5  h true     (4.3s)  h: r_eff=  0 Z=2.26  hu: r_eff=  9 Z=3.81  huv: r_eff= 29 Z=30.95
  U0=0.5   T=5  h uniform  (4.2s)  h: r_eff=  0 Z=2.21  hu: r_eff=  9 Z=3.84  huv: r_eff= 29 Z=31.84
  U0=0.5   T=12 h true     (4.3s)  h: r_eff=  0 Z=1.17  hu: r_eff=  7 Z=5.79  huv: r_eff= 34 Z=37.02
  U0=0.5   T=12 h uniform  (4.1s)  h: r_eff=  0 Z=1.20  hu: r_eff=  7 Z=5.70  huv: r_eff= 34 Z=40.00

[Cell 5] baseline (T=5h,U0=0.15): Z_h=0.68  Z_hu=1.54  Z_huv=6.73
[Cell 5] قاعده: Z_huv<3 => فرضیه‌ی «ب» تأیید (طراحی دوقلو ضعیف است)؛ T/U0 را تغییر دهید.
[Cell 5] 

In [6]:
# ============================================================
# Cell 6 — معماری دوشاخه (خروجی‌های مقیاس‌دار، ویژگی‌های فوریه‌ی سازگار با شبکه، φ₀ هموار)
# ============================================================
def init_mlp(key, layer_sizes):
    params = []
    for k, n_in, n_out in zip(jax.random.split(key, len(layer_sizes) - 1), layer_sizes[:-1], layer_sizes[1:]):
        W = jax.random.normal(k, (n_in, n_out)) * jnp.sqrt(2.0 / n_in)
        params.append((W, jnp.zeros(n_out)))
    return params

def mlp_apply(params, x):
    a = x
    for i, (W, b) in enumerate(params):
        z = a @ W + b
        a = jnp.tanh(z) if i < len(params) - 1 else z
    return a

N_FOURIER = 4
def fourier_features(xy, n_freq=N_FOURIER, scale=1.0):
    # فرکانس بیشینه n_freq*scale*pi => n_freq*scale دوره روی دامنه؛ باید << NX/2 بماند (نایکوئیست)
    freqs = scale * jnp.arange(1, n_freq + 1) * jnp.pi
    ang = (xy[..., None] * freqs).reshape(xy.shape[:-1] + (-1,))
    return jnp.concatenate([xy, jnp.sin(ang), jnp.cos(ang)], axis=-1)

HYDRO_LAYERS = [5, 64, 64, 64, 3]               # (x,y,t, sin(wt), cos(wt)) -> (h,u,v)/scale
PHI_IN_DIM = 2 + 2 * 2 * N_FOURIER
PHI_LAYERS = [PHI_IN_DIM, 32, 32, 1]

def _inv_softplus(y):
    return jnp.log(jnp.expm1(y))

def init_params():
    k1, k2 = next_key(), next_key()
    phi = init_mlp(k2, PHI_LAYERS)
    W, b = phi[-1]; phi[-1] = (0.1 * W, b)       # φ₀ کوچک => σ(kφ)≈0.5
    return {"hydro": init_mlp(k1, HYDRO_LAYERS), "phi_net": phi,
            "cf_low_raw": _inv_softplus(jnp.array(0.7)),
            "cf_high_raw": _inv_softplus(jnp.array(1.3))}

def get_cf_low_high(params):
    return (CF_BACKGROUND * jax.nn.softplus(params["cf_low_raw"]),
            CF_BACKGROUND * jax.nn.softplus(params["cf_high_raw"]))

def hydro_apply(params, x, y, t):
    feat = jnp.stack([x / LX, y / LY, t / T_TOTAL, jnp.sin(OMEGA * t), jnp.cos(OMEGA * t)], axis=-1)
    out = mlp_apply(params["hydro"], feat)
    return H_SC * out[..., 0], U_SC * out[..., 1], U_SC * out[..., 2]

def phi_apply(params, x, y):
    feat = fourier_features(jnp.stack([x / LX, y / LY], axis=-1))
    return mlp_apply(params["phi_net"], feat)[..., 0]

def cf_apply(params, x, y, k):
    sig = jax.nn.sigmoid(k * phi_apply(params, x, y))
    lo, hi = get_cf_low_high(params)
    return lo + (hi - lo) * sig

def cf_apply_kzone_soft(params_kzone, x, y, temperature):
    # گزینه‌ی ب (فاز ۲/H1d): تخصیص نرم K-ناحیه‌ای. params_kzone: {'zone_net','cf_values'}
    logits = mlp_apply(params_kzone["zone_net"], jnp.stack([x / LX, y / LY], axis=-1))
    w = jax.nn.softmax(logits / temperature, axis=-1)
    return jnp.sum(w * params_kzone["cf_values"][None, :], axis=-1)

_P0 = init_params()
print("[Cell 6] hydro:", sum(w.size + b.size for w, b in _P0["hydro"]), "| phi:", sum(w.size + b.size for w, b in _P0["phi_net"]),
      "| Cf0:", [float(c) for c in get_cf_low_high(_P0)])

[Cell 6] hydro: 8899 | phi: 1697 | Cf0: [0.0017499999999999998, 0.0032500000000000003]


In [7]:
# ============================================================
# Cell 7 — هزینه (بی‌بعد)، IC/BC، Adam با lr جداگانه و ε=1e-20، آموزش دو-فازی Freeze-Release
# ============================================================
def pde_residual_batch(params, x, y, t, k, wind_on, diff_on):
    f = lambda xx, yy, tt: jnp.stack(hydro_apply(params, xx, yy, tt))     # (3,) = (h,u,v)
    def single(x_, y_, t_):
        dx, dy, dt = jax.jacfwd(f, argnums=(0, 1, 2))(x_, y_, t_)         # هر کدام (3,)
        h_, u_, v_ = f(x_, y_, t_)
        h_t, u_t, v_t = dt[0], dt[1], dt[2]
        h_x, h_y, u_x, v_y = dx[0], dy[0], dx[1], dy[2]
        cf_ = cf_apply(params, x_[None], y_[None], k)[0]
        speed = jnp.sqrt(u_**2 + v_**2 + 1e-8)
        wind_x = WIND_AMPLITUDE * jnp.sin(2 * jnp.pi * y_ / LY) if wind_on else 0.0
        if diff_on:
            d2 = lambda a: jax.jacfwd(lambda z: jax.jacfwd(f, argnums=a)(*( (z, y_, t_) if a == 0 else (x_, z, t_) )))
            fxx = d2(0)(x_); fyy = d2(1)(y_)                               # (3,)
            diff_u = NU_DIFF * (fxx[1] + fyy[1]); diff_v = NU_DIFF * (fxx[2] + fyy[2])
        else:
            diff_u = diff_v = 0.0
        res_u = u_t + G * h_x - F0 * v_ + (cf_ / H0) * u_ * speed - wind_x - diff_u
        res_v = v_t + G * h_y + F0 * u_ + (cf_ / H0) * v_ * speed - diff_v
        res_c = h_t + H0 * (u_x + v_y)
        return res_u, res_v, res_c
    ru, rv, rc = jax.vmap(single)(x, y, t)
    return ru / A_REF, rv / A_REF, rc / C_REF

def velocity_cap_penalty(params):
    _, u_c, v_c = hydro_apply(params, COLL_X, COLL_Y, COLL_T)
    return jnp.mean(jax.nn.relu(jnp.abs(u_c) - VELOCITY_REG_CAP_MPS)**2) + jnp.mean(jax.nn.relu(jnp.abs(v_c) - VELOCITY_REG_CAP_MPS)**2)

def zone_separation_penalty(params):
    lo, hi = get_cf_low_high(params)
    return jax.nn.relu(ZONE_SEPARATION_MARGIN - jnp.abs(hi - lo) / 1e-3) ** 2

def icbc_loss(params):
    z = jnp.zeros_like
    h0, u0_, v0_ = hydro_apply(params, IC_X, IC_Y, z(IC_X))
    uW = hydro_apply(params, z(BCW_Y), BCW_Y, BCW_T)[1]
    uE = hydro_apply(params, z(BCE_Y) + LX, BCE_Y, BCE_T)[1]
    vS = hydro_apply(params, BCS_X, z(BCS_X), BCS_T)[2]
    vN = hydro_apply(params, BCN_X, z(BCN_X) + LY, BCN_T)[2]
    tgtW = U0 * jnp.cos(OMEGA * (BCW_T - DT))     # u(x=0,t=(n+1)DT) = U0 cos(w n DT) در حل‌گر
    return (jnp.mean((h0 / H_SC)**2) + jnp.mean((u0_ / U_SC)**2) + jnp.mean((v0_ / U_SC)**2)
            + jnp.mean(((uW - tgtW) / U_SC)**2) + jnp.mean((uE / U_SC)**2)
            + jnp.mean((vS / U_SC)**2) + jnp.mean((vN / U_SC)**2))

def build_loss_and_grad(wind_on, diff_on, observe_velocity, h_obs, u_obs, v_obs, sigs):
    # factory: هر سناریو یک تابع jit تازه (closure روی پرچم‌ها/داده‌ها)
    n_snap = h_obs.shape[0]
    fx, fy, ft = jnp.tile(OBS_X, (n_snap,)), jnp.tile(OBS_Y, (n_snap,)), jnp.repeat(OBS_T, N_OBS_PTS)
    sh, su, sv = sigs
    def data_loss(params):
        h, u, v = hydro_apply(params, fx, fy, ft)
        L = jnp.mean(((h.reshape(n_snap, -1) - h_obs) / sh) ** 2)
        if observe_velocity:
            L = L + jnp.mean(((u.reshape(n_snap, -1) - u_obs) / su) ** 2) + jnp.mean(((v.reshape(n_snap, -1) - v_obs) / sv) ** 2)
        return L
    def physics_loss(params, k):
        ru, rv, rc = pde_residual_batch(params, COLL_X, COLL_Y, COLL_T, k, wind_on, diff_on)
        return jnp.mean(ru**2) + jnp.mean(rv**2) + jnp.mean(rc**2)
    def total_loss(params, k, phys_w):
        L = data_loss(params) + phys_w * physics_loss(params, k) + velocity_cap_penalty(params) \
            + ZONE_SEPARATION_WEIGHT * zone_separation_penalty(params)
        if USE_ICBC:
            L = L + ICBC_WEIGHT * icbc_loss(params)
        return L
    return jax.jit(jax.value_and_grad(total_loss))

_tm = jax.tree_util.tree_map
def adam_init(params):
    return {"m": _tm(jnp.zeros_like, params), "v": _tm(jnp.zeros_like, params), "t": 0}

def make_lr_tree(params, lr_hydro, lr_phi, lr_cf):
    return {"hydro": _tm(lambda _: lr_hydro, params["hydro"]), "phi_net": _tm(lambda _: lr_phi, params["phi_net"]),
            "cf_low_raw": lr_cf, "cf_high_raw": lr_cf}

@jax.jit
def adam_step(params, grads, state, lr_tree, eps=1e-20):
    t = state["t"] + 1
    b1, b2 = 0.9, 0.999
    m = _tm(lambda m_, g: b1 * m_ + (1 - b1) * g, state["m"], grads)
    v = _tm(lambda v_, g: b2 * v_ + (1 - b2) * g**2, state["v"], grads)
    new = _tm(lambda p, m_, v_, lr: p - lr * (m_ / (1 - b1**t)) / (jnp.sqrt(v_ / (1 - b2**t)) + eps), params, m, v, lr_tree)
    return new, {"m": m, "v": v, "t": t}

def decay_moments(state, keys, decay):
    m, v = dict(state["m"]), dict(state["v"])
    for k_ in keys:
        m[k_] = _tm(lambda a: a * decay, m[k_]); v[k_] = _tm(lambda a: a * decay, v[k_])
    return {"m": m, "v": v, "t": state["t"]}

def _norm(tree):
    return float(jnp.sqrt(sum(jnp.sum(l**2) for l in jax.tree_util.tree_leaves(tree))))
def grad_norms(g):
    return dict(g_hydro=_norm(g["hydro"]), g_phi=_norm(g["phi_net"]), g_cf=_norm((g["cf_low_raw"], g["cf_high_raw"])))

def train(params, loss_and_grad, lr_hydro=1e-3, lr_phi=1e-3, lr_cf=3e-3, log_every=None):
    total = ADAM_EPOCHS_PHASE1 + ADAM_EPOCHS_PHASE2
    log_every = log_every or max(1, total // 10)
    state = adam_init(params)
    lr1 = make_lr_tree(params, lr_hydro, 0.0, 0.0)           # فاز ۱: شاخه‌ی اصطکاک freeze
    lr2 = make_lr_tree(params, lr_hydro, lr_phi, lr_cf)      # فاز ۲: release
    history, log = [], []
    def _log(phase, ep, loss, k, g):
        lo, hi = get_cf_low_high(params); gn = grad_norms(g)
        row = dict(phase=phase, epoch=ep, loss=float(loss), k=float(k), cf_low=float(lo), cf_high=float(hi), **gn,
                   ratio_phi_hydro=gn["g_phi"] / max(gn["g_hydro"], 1e-300))
        log.append(row)
        print(f"  فاز{phase} ep{ep:6d} loss={row['loss']:.4e} k={row['k']:.1f} Cf=({row['cf_low']:.5f},{row['cf_high']:.5f}) "
              f"|g_hyd|={gn['g_hydro']:.1e} |g_phi|={gn['g_phi']:.1e} |g_cf|={gn['g_cf']:.1e} ratio={row['ratio_phi_hydro']:.1e}")
    print("[Cell 7] فاز ۱ (freeze اصطکاک، وزن فیزیک کم) ...")
    for ep in range(ADAM_EPOCHS_PHASE1):
        loss, g = loss_and_grad(params, K_MIN, PHYS_WEIGHT_PHASE1)
        params, state = adam_step(params, g, state, lr1)
        history.append(float(loss))
        if ep % log_every == 0: _log(1, ep, loss, K_MIN, g)
    print("[Cell 7] فاز ۲ (release + annealing k) ...")
    for ep in range(ADAM_EPOCHS_PHASE2):
        if ep < WARM_TRANSITION_EPOCHS:
            state = decay_moments(state, ("phi_net", "cf_low_raw", "cf_high_raw"), MOMENT_DECAY_RATE)
        k = k_schedule(ep)
        loss, g = loss_and_grad(params, k, PHYS_WEIGHT_PHASE2)
        params, state = adam_step(params, g, state, lr2)
        history.append(float(loss))
        if ep % log_every == 0: _log(2, ep, loss, k, g)
    return params, history, log
print("[Cell 7] آماده.")

[Cell 7] آماده.


In [8]:
# ============================================================
# Cell 8 — L-BFGS-B (full-batch) با تحمل‌های سخت‌گیرانه‌تر
# ============================================================
from scipy.optimize import minimize

def flatten_params(params):
    leaves, treedef = jax.tree_util.tree_flatten(params)
    shapes = [l.shape for l in leaves]
    return np.concatenate([np.asarray(l).ravel() for l in leaves]), treedef, shapes

def unflatten_params(flat, treedef, shapes):
    leaves, idx = [], 0
    for shp in shapes:
        n = int(np.prod(shp)) if shp else 1
        leaves.append(jnp.array(flat[idx:idx + n]).reshape(shp)); idx += n
    return jax.tree_util.tree_unflatten(treedef, leaves)

def lbfgs_finetune(params, loss_and_grad, k_final, maxiter=None):
    maxiter = maxiter or LBFGS_MAXITER
    flat0, treedef, shapes = flatten_params(params)
    def fn(flat):
        loss, g = loss_and_grad(unflatten_params(flat, treedef, shapes), k_final, PHYS_WEIGHT_PHASE2)
        return float(loss), np.asarray(flatten_params(g)[0], dtype=np.float64)
    print(f"[Cell 8] L-BFGS-B (حداکثر {maxiter} تکرار) ...")
    res = minimize(fn, flat0, jac=True, method="L-BFGS-B",
                   options=dict(maxiter=maxiter, maxfun=5 * maxiter, maxcor=30, ftol=1e-14, gtol=1e-10))
    print(f"[Cell 8] پایان: loss={res.fun:.4e} nit={res.nit} | {res.message}")
    return unflatten_params(res.x, treedef, shapes), dict(nit=int(res.nit), message=str(res.message), loss=float(res.fun))

In [9]:
# ============================================================
# Cell 9 — ارزیابی: RMSE/r، AUC (خام و sign-agnostic)، خطای مرز (میانگین/Hausdorff)، خطای (h,u,v)، کنترل باد
# ============================================================
_XC = (jnp.arange(NX) + 0.5) * DX; _YC = (jnp.arange(NY) + 0.5) * DY
_XG, _YG = jnp.meshgrid(_XC, _YC, indexing="ij")

def predicted_cf_field(params, k_final):
    return cf_apply(params, _XG.ravel(), _YG.ravel(), k_final).reshape(NX, NY)

def rmse_r_metrics(cf_pred, cf_true):
    p, t = np.asarray(cf_pred), np.asarray(cf_true)
    rmse = 100 * np.sqrt(np.mean((p - t) ** 2)) / (t.max() - t.min())
    r = np.corrcoef(p.ravel(), t.ravel())[0, 1] if p.std() > 0 else float("nan")
    return rmse, r

def _boundary_cells(lbl):
    e = np.zeros_like(lbl, dtype=bool)
    e[:-1, :] |= lbl[:-1, :] != lbl[1:, :]
    e[:, :-1] |= lbl[:, :-1] != lbl[:, 1:]
    return np.argwhere(e)

def zone_scores(cf_pred, cf_true, thr_pred):
    p, t = np.asarray(cf_pred), np.asarray(cf_true)
    tl = (t > 0.5 * (CF_LOW_TRUE + CF_HIGH_TRUE)).astype(int)
    pos, neg = p.ravel()[tl.ravel() == 1], p.ravel()[tl.ravel() == 0]
    auc = float(np.mean(pos[:, None] > neg[None, :]) + 0.5 * np.mean(pos[:, None] == neg[None, :]))
    te, pe = _boundary_cells(tl), _boundary_cells((p > thr_pred).astype(int))
    if len(te) == 0 or len(pe) == 0:
        return auc, max(auc, 1 - auc), float("nan"), float("nan")
    d = np.sqrt(((pe[:, None, :] - te[None, :, :]) ** 2).sum(-1))       # واحد: اندیس سلول
    mean_err = 0.5 * (d.min(1).mean() + d.min(0).mean())
    hausdorff = max(d.min(1).max(), d.min(0).max())
    return auc, max(auc, 1 - auc), float(mean_err), float(hausdorff)

def hydro_state_error(params, truth):
    out = {}
    for name, arr, idx in (("h", truth[0], 0), ("u", truth[1], 1), ("v", truth[2], 2)):
        a = np.asarray(arr); rng = a.max() - a.min()
        errs = []
        for s in SNAP:
            pred = hydro_apply(params, _XG.ravel(), _YG.ravel(), jnp.full(_XG.size, (s + 1) * DT))[idx].reshape(NX, NY)
            errs.append(np.mean(np.abs(np.asarray(pred) - a[s])))
        out[f"mae_{name}_pct"] = 100 * float(np.mean(errs)) / rng
    return out

def wind_correlation_controls(cf_pred, cf_true, rng):
    w = np.asarray(jnp.tile(WIND_PATTERN_U[0][None, :], (NX, 1)))
    err = np.asarray(cf_pred) - np.asarray(cf_true)
    c = lambda a: float(np.corrcoef(np.ravel(a), w.ravel())[0, 1]) if np.std(a) > 0 else float("nan")
    return c(err), c(cf_true), c(rng.normal(size=w.shape))

def evaluate(params, k_final, truth, meta):
    cf_pred = predicted_cf_field(params, k_final)
    lo, hi = [float(c) for c in get_cf_low_high(params)]
    rmse, r = rmse_r_metrics(cf_pred, CF_TRUE)
    auc, auc_sym, berr, bhaus = zone_scores(cf_pred, CF_TRUE, 0.5 * (lo + hi))
    ce, ct, cr = wind_correlation_controls(cf_pred, CF_TRUE, np.random.default_rng(5))
    res = dict(meta, patch_level=PATCH_LEVEL, run_mode=RUN_MODE, T_hours=DESIGN_T_HOURS, U0=U0,
               rmse_pct=float(rmse), r=float(r), auc=auc, auc_sym=auc_sym, boundary_mean_cells=berr, boundary_hausdorff_cells=bhaus,
               corr_err_wind=ce, corr_true_wind=ct, corr_random_wind=cr, cf_low=lo, cf_high=hi, **hydro_state_error(params, truth))
    print(f"[Cell 9] {meta['label']} (seed {meta['seed']}): RMSE={rmse:.1f}% r={r:.3f} AUC={auc:.3f}(sym {auc_sym:.3f}) "
          f"bnd={berr:.2f}/{bhaus:.2f} cells | Cf=({lo:.4g},{hi:.4g}) true=({CF_LOW_TRUE:.4g},{CF_HIGH_TRUE:.4g}) | "
          f"MAE% h/u/v={res['mae_h_pct']:.1f}/{res['mae_u_pct']:.1f}/{res['mae_v_pct']:.1f}")
    return res, np.asarray(cf_pred)

In [10]:
# ============================================================
# Cell 10 — ارکستراسیون: بازوها × seedها، معیار پذیرش، ذخیره‌ی نتایج و شکل‌ها
# ============================================================
ARMS = {
    "control":  dict(wind_on=True,  diff_on=True,  observe_velocity=True,  label="CONTROL h+u+v"),
    "honly_ww": dict(wind_on=False, diff_on=False, observe_velocity=False, label="h-only wind OFF diff OFF"),
    "honly_wo": dict(wind_on=True,  diff_on=False, observe_velocity=False, label="h-only wind ON diff OFF"),
    "honly_wd": dict(wind_on=True,  diff_on=True,  observe_velocity=False, label="h-only wind ON diff ON"),
}

def run_scenario(arm, seed):
    cfg = ARMS[arm]
    set_seed(seed)
    (h_obs, u_obs, v_obs), sigs, truth = make_observations(cfg["wind_on"], cfg["diff_on"], 1000 + seed)
    params = init_params()
    cf0 = [float(c) for c in get_cf_low_high(params)]
    lg = build_loss_and_grad(cfg["wind_on"], cfg["diff_on"], cfg["observe_velocity"], h_obs, u_obs, v_obs, sigs)
    t0 = time.time()
    params, history, log = train(params, lg)
    params, lb = lbfgs_finetune(params, lg, K_MAX)
    res, cf_pred = evaluate(params, K_MAX, truth, dict(arm=arm, label=cfg["label"], seed=seed, cf0=cf0, lbfgs=lb,
                                                         wall_s=time.time() - t0))
    return res, cf_pred, log, history

def acceptance_control(res, log):
    ph2 = [r["ratio_phi_hydro"] for r in log if r["phase"] == 2]
    ratio = float(np.median(ph2)) if ph2 else float("nan")
    d0 = abs(res["cf_low"] - CF_LOW_TRUE) + abs(res["cf_high"] - CF_HIGH_TRUE)
    d1 = abs(res["cf0"][0] - CF_LOW_TRUE) + abs(res["cf0"][1] - CF_HIGH_TRUE)
    checks = {"ratio |g_phi|/|g_hydro| >= 1e-3": ratio >= 1e-3, "Cf scalars moved toward truth": d0 < 0.5 * d1,
              "r(Cf) > 0.5": res["r"] > 0.5, "AUC > 0.85": res["auc"] > 0.85}
    print(f"[Accept] median ratio(phase2)={ratio:.2e}")
    for k_, ok in checks.items(): print(f"   {'PASS' if ok else 'FAIL'}  {k_}")
    return all(checks.values())

def _fig(res, cf_pred, path):
    fig, ax = plt.subplots(1, 3, figsize=(13, 3))
    for a, (arr, ttl) in zip(ax, ((CF_TRUE, "Cf true"), (cf_pred, "Cf pred"), (cf_pred - np.asarray(CF_TRUE), "error"))):
        im = a.imshow(np.asarray(arr).T, origin="lower", aspect="auto"); a.set_title(ttl); plt.colorbar(im, ax=a)
    fig.suptitle(f"{res['label']} seed={res['seed']} r={res['r']:.2f} AUC={res['auc']:.2f}")
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.close(fig)

ALL_RESULTS = []
if RUN_TRAINING:
    for arm in TRAIN_ARMS:
        for seed in SEEDS:
            print(f"\n{'='*70}\n[Cell 10] arm={arm} seed={seed} mode={RUN_MODE}\n{'='*70}")
            res, cf_pred, log, hist = run_scenario(arm, seed)
            ALL_RESULTS.append(res)
            tag = f"{arm}_s{seed}_{RUN_MODE}"
            _fig(res, cf_pred, os.path.join(OUT_DIR, f"cf_{tag}.png"))
            with open(os.path.join(OUT_DIR, f"log_{tag}.json"), "w") as fh: json.dump(log, fh)
            if arm == "control": res["accepted"] = acceptance_control(res, log)
            with open(os.path.join(OUT_DIR, f"results_{RUN_MODE}.json"), "w") as fh: json.dump(ALL_RESULTS, fh, indent=1, default=float)
    print("\n" + "=" * 100)
    print(f"{'arm':<10}{'seed':>5}{'RMSE%':>8}{'r':>8}{'AUC':>8}{'AUCsym':>8}{'bnd':>7}{'MAEu%':>8}")
    for r in ALL_RESULTS:
        print(f"{r['arm']:<10}{r['seed']:>5}{r['rmse_pct']:>8.1f}{r['r']:>8.3f}{r['auc']:>8.3f}{r['auc_sym']:>8.3f}"
              f"{r['boundary_mean_cells']:>7.2f}{r['mae_u_pct']:>8.1f}")
    if any(r["arm"] == "control" and not r.get("accepted", False) for r in ALL_RESULTS):
        print("\n[!] بازوی کنترل پذیرفته نشد: بازوهای فقط-h را اجرا نکنید؛ ابتدا Runbook بخش «درخت تصمیم» را ببینید.")


[Cell 10] arm=control seed=0 mode=smoke
[Cell 7] فاز ۱ (freeze اصطکاک، وزن فیزیک کم) ...
  فاز1 ep     0 loss=6.8219e+05 k=3.0 Cf=(0.00175,0.00325) |g_hyd|=6.2e+06 |g_phi|=4.4e-04 |g_cf|=2.9e-04 ratio=7.2e-11
  فاز1 ep    10 loss=6.8336e+04 k=3.0 Cf=(0.00175,0.00325) |g_hyd|=1.6e+06 |g_phi|=1.0e-04 |g_cf|=6.3e-05 ratio=6.4e-11
  فاز1 ep    20 loss=1.0459e+04 k=3.0 Cf=(0.00175,0.00325) |g_hyd|=6.0e+05 |g_phi|=1.1e-05 |g_cf|=7.6e-06 ratio=1.8e-11
  فاز1 ep    30 loss=1.0000e+04 k=3.0 Cf=(0.00175,0.00325) |g_hyd|=4.5e+05 |g_phi|=3.6e-05 |g_cf|=2.0e-05 ratio=8.1e-11
  فاز1 ep    40 loss=5.1610e+03 k=3.0 Cf=(0.00175,0.00325) |g_hyd|=2.6e+05 |g_phi|=3.1e-05 |g_cf|=7.1e-06 ratio=1.2e-10
[Cell 7] فاز ۲ (release + annealing k) ...
  فاز2 ep     0 loss=2.3498e+03 k=3.0 Cf=(0.00175,0.00325) |g_hyd|=6.6e+04 |g_phi|=4.4e-03 |g_cf|=1.4e-03 ratio=6.7e-08
  فاز2 ep    10 loss=2.0977e+03 k=7.4 Cf=(0.00172,0.00327) |g_hyd|=4.3e+04 |g_phi|=4.9e-03 |g_cf|=1.8e-03 ratio=1.2e-07
  فاز2 ep    20 loss=2.0331

In [11]:
# ============================================================
# Cell 11 (اختیاری، P1) — کنترل کلاسیک: Tikhonov + L-BFGS-B مستقیم روی حل‌گر (بدون شبکه‌ی عصبی)
# ============================================================
# RUN_CLASSICAL=True. گرادیان با reverse-mode از میان حل‌گر؛ پارامتر = میدان Cf در واحد 1e-3؛ regularizer = گرادیان مکانی (Tikhonov مرتبه‌ی ۱).
# توجه: «inverse crime» عمدی است (همان حل‌گر برای داده و وارون) -> این بهترین حالت ممکن برای کنترل است.
from scipy.optimize import minimize as _minimize

def classical_inversion(channels, lam, seed=0, iters=300, wind_on=True, diff_on=True):
    ch = tuple(channels)
    clean = forward_obs(CF_TRUE, wind_on, diff_on, N_STEPS, U0, SNAP, IXO, IYO, ch)          # (n_snap,n_ch,n_obs)
    sig = OBS_NOISE_STD_FRAC * jnp.std(clean, axis=(0, 2))
    rng = np.random.default_rng(2000 + seed)
    obs = clean + jnp.array(rng.normal(size=clean.shape)) * sig[None, :, None]
    def loss(x):
        xf = x.reshape(NX, NY)
        y = forward_obs(xf * 1e-3, wind_on, diff_on, N_STEPS, U0, SNAP, IXO, IYO, ch)
        mis = jnp.sum(((y - obs) / sig[None, :, None]) ** 2)
        reg = lam * (jnp.sum(jnp.diff(xf, axis=0)**2) + jnp.sum(jnp.diff(xf, axis=1)**2))
        return mis + reg
    vg = jax.jit(jax.value_and_grad(loss))
    x0 = np.full(NX * NY, CF_BACKGROUND / 1e-3)
    fn = lambda z: (lambda lv, gv: (float(lv), np.asarray(gv, dtype=np.float64)))(*vg(jnp.array(z)))
    res = _minimize(fn, x0, jac=True, method="L-BFGS-B", bounds=[(0.2, 10.0)] * (NX * NY),
                    options=dict(maxiter=iters, maxfun=5 * iters, ftol=1e-14, gtol=1e-10))
    cf = res.x.reshape(NX, NY) * 1e-3
    rmse, r = rmse_r_metrics(cf, CF_TRUE)
    auc, auc_sym, berr, bh = zone_scores(cf, CF_TRUE, 0.5 * (cf.min() + cf.max()))
    return dict(channels="".join(ch), lam=lam, nit=int(res.nit), misfit_at_truth_expected=int(clean.size),
                loss=float(res.fun), rmse_pct=float(rmse), r=float(r), auc=auc, auc_sym=auc_sym, boundary_mean_cells=berr)

CLASSICAL_ROWS = []
if RUN_CLASSICAL:
    for chs in (("h",), ("h", "u", "v")):
        for lam in (1e-2, 1.0, 1e2):
            row = classical_inversion(chs, lam, iters=RC["lbfgs_iters"])
            CLASSICAL_ROWS.append(row); print(row)
    with open(os.path.join(OUT_DIR, f"classical_{RUN_MODE}.json"), "w") as fh: json.dump(CLASSICAL_ROWS, fh, indent=1)
    print("[Cell 11] تفسیر: اگر h+u+v با λ مناسب r>0.5 دهد ولی h-only نه => شناسایی‌ناپذیری مستقل از PINN.")